https://medium.com/@mygreatlearning/everything-you-need-to-know-about-vgg16-7315defb5918

In [ ]:
import torch
from torch import nn
from torch.nn import Conv2d, MaxPool2d, Linear, Flatten, Softmax, CrossEntropyLoss, BCEWithLogitsLoss
import numpy as np
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.optim.sgd import SGD
from torchinfo import summary
from torchvision.transforms import v2
from torch.optim import Adam
import pandas as pd
from sklearn.metrics import accuracy_score

In [ ]:
data = pd.read_csv("../data/chest_xray/chest_xray_dataset.csv")
device = torch.device('cuda') if torch.cuda.is_available else torch.device('cpu')

In [ ]:
from PIL import Image
import os 

class XrayDataset(Dataset):
    def __init__(self, split, transform):
        self.data = data[data['split'] == split]

        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        data_item = self.data.iloc[idx]
        label = self.data.iloc[idx]['class']
        image_data = Image.open(os.path.join("..", data_item['path']))
        image_data = self.transform(image_data)
        return image_data, label

In [ ]:
train_transform = v2.Compose([
    v2.ToImage(),
    v2.Grayscale(num_output_channels=1),
    v2.ToDtype(torch.uint8, scale=True),
    v2.RandomResizedCrop(size=(500, 500)),
    v2.RandomHorizontalFlip(),
    v2.ToDtype(torch.float32, scale=True),
])

test_transform = v2.Compose([
    v2.ToImage(),
    v2.Grayscale(num_output_channels=1),
    v2.ToDtype(torch.uint8, scale=True),
    v2.CenterCrop(size=(500, 500)),
    v2.ToDtype(torch.float32, scale=True),
])

In [ ]:
train_dataset = XrayDataset(split='train', transform=train_transform)
val_dataset = XrayDataset(split="val", transform=test_transform)
test_dataset = XrayDataset(split="test", transform=test_transform)

train_dataloader = DataLoader(dataset=train_dataset, shuffle=True, batch_size=64)
val_dataloader = DataLoader(dataset=val_dataset, shuffle=False, batch_size=64)
test_dataloader = DataLoader(dataset=test_dataset, shuffle=False, batch_size=64)

In [ ]:
class VGG16(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1_1 = Conv2d(in_channels=1, out_channels=64, kernel_size=(3,3), padding='same')
        self.conv1_2 = Conv2d(in_channels=64, out_channels=64, kernel_size=(3,3), padding='same')
        self.pooling1 = MaxPool2d(kernel_size=(2,2), stride=2)
        self.conv2_1 = Conv2d(in_channels=64, out_channels=128, kernel_size=(3,3), padding='same')
        self.conv2_2 = Conv2d(in_channels=128, out_channels=128, kernel_size=(3,3), padding='same')
        self.pooling2 = MaxPool2d(kernel_size=(2,2), stride=2)
        self.conv3_1 = Conv2d(in_channels=128, out_channels=256, kernel_size=(3,3), padding='same')
        self.conv3_2 = Conv2d(in_channels=256, out_channels=256, kernel_size=(3,3), padding='same')
        self.conv3_3 = Conv2d(in_channels=256, out_channels=256, kernel_size=(3,3), padding='same')
        self.pooling3 = MaxPool2d(kernel_size=(2,2), stride=2)
        self.conv4_1 = Conv2d(in_channels=256, out_channels=512, kernel_size=(3,3), padding='same')
        self.conv4_2 = Conv2d(in_channels=512, out_channels=512, kernel_size=(3,3), padding='same')
        self.conv4_3 = Conv2d(in_channels=512, out_channels=512, kernel_size=(3,3), padding='same')
        self.pooling4 = MaxPool2d(kernel_size=(2,2), stride=2)
        self.conv5_1 = Conv2d(in_channels=512, out_channels=512, kernel_size=(3,3), padding='same')
        self.conv5_2 = Conv2d(in_channels=512, out_channels=512, kernel_size=(3,3), padding='same')
        self.conv5_3 = Conv2d(in_channels=512, out_channels=512, kernel_size=(3,3), padding='same')
        self.pooling5 = MaxPool2d(kernel_size=(2,2), stride=2)
        self.flatten = Flatten()
        self.fc1 = Linear(7*7*512, 4096)
        self.fc2 = Linear(4096, 4096)
        self.fc3 = Linear(4096, 1000)
        self.softmax = Softmax()
        
    def forward(self, x):
        x = self.conv1_1(x)
        x = self.conv1_2(x)
        x = self.pooling1(x)
        x = self.conv2_1(x)
        x = self.conv2_2(x)
        x = self.pooling2(x)
        x = self.conv3_1(x)
        x = self.conv3_2(x)
        x = self.conv3_3(x)
        x = self.pooling3(x)
        x = self.conv4_1(x)
        x = self.conv4_2(x)
        x = self.conv4_3(x)
        x = self.pooling4(x)
        x = self.conv5_1(x)
        x = self.conv5_2(x)
        x = self.conv5_3(x)
        x = self.pooling5(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        # x = self.softmax(x)
       
        return x

In [16]:
model = VGG16()

dummy = torch.tensor(np.zeros(shape=(16, 3, 224, 224))).float()

summary(model=model, input_data=dummy)

torch.Size([16, 25088])


Layer (type:depth-idx)                   Output Shape              Param #
VGG16                                    [16, 1000]                --
├─Conv2d: 1-1                            [16, 64, 224, 224]        1,792
├─Conv2d: 1-2                            [16, 64, 224, 224]        36,928
├─MaxPool2d: 1-3                         [16, 64, 112, 112]        --
├─Conv2d: 1-4                            [16, 128, 112, 112]       73,856
├─Conv2d: 1-5                            [16, 128, 112, 112]       147,584
├─MaxPool2d: 1-6                         [16, 128, 56, 56]         --
├─Conv2d: 1-7                            [16, 256, 56, 56]         295,168
├─Conv2d: 1-8                            [16, 256, 56, 56]         590,080
├─Conv2d: 1-9                            [16, 256, 56, 56]         590,080
├─MaxPool2d: 1-10                        [16, 256, 28, 28]         --
├─Conv2d: 1-11                           [16, 512, 28, 28]         1,180,160
├─Conv2d: 1-12                           [16, 5

In [ ]:
epochs = 15
criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(model.parameters(), 0.0001)

for epoch in range(epochs):
    epoch_loss = 0
    print(f"Epoch {epoch}:")
    model.train()
    for idx, (x, target) in enumerate(train_dataloader):
        input = x.to(device)
        target = target.to(device)
        output = model(input)
        output = output.squeeze()
        loss = criterion(output, target.float())
        # zero the gradients
        optimizer.zero_grad()
        # calculate the gradients based on the calculated loss (the gradient of the loss wrt each param)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    # Validation loss
    print("Loss", epoch_loss/len(train_dataloader))
    
    model.eval()
    with torch.no_grad():
        val_loss = 0
        all_preds = []
        all_targets = []

        for x, target in val_dataloader:
            x = x.to(device)
            target = target.to(device).float()

            output = model(x).squeeze()

            loss = criterion(output, target)
            val_loss += loss.item()

            preds = torch.sigmoid(output)
            all_preds.extend(preds.cpu())
            all_targets.extend(target.cpu())

        all_preds = torch.stack(all_preds)
        all_targets = torch.stack(all_targets)

        output_thresholded = (all_preds >= 0.5).numpy().astype(int)
        print(output_thresholded)
        accuracy = accuracy_score(all_targets.numpy(), output_thresholded)

        print(f"Val loss: {val_loss / len(val_dataloader)} | Val accuracy: {accuracy}")